In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Set absolute path for new metastore
new_metastore_path = os.path.abspath('/apps/sandbox/metastore')
metastore_url = f"jdbc:derby:;databaseName={new_metastore_path};create=true"

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.sql.catalogImplementation", "hive")
    .set("spark.sql.catalog.spark_catalog.warehouse","s3a://warehouse/default")
    .set("spark.sql.warehouse.dir", "s3a://warehouse/default/spark")
    .set("javax.jdo.option.ConnectionURL", metastore_url)
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("08-zorder").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

:: loading settings :: url = jar:file:/opt/spark-3.5.3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brijeshdhaker/.ivy2/cache
The jars for the packages stored in: /home/brijeshdhaker/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f5e839f3-d2bf-4066-9f74-af5147dac8b0;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in local-m2-cache
	found io.delta#delta-storage;3.3.2 in local-m2-cache
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
	found org.apache.hadoop#hadoop-aws;3.3.4 in local-m2-cache
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in local-m2-cache
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 188ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from local-m2-cache in [default]
	io.delta#delta-spark_2.12;3.3.2 from local-m2-cache in [default]
	io.delta#delta-storage;3.3.2 from local-m2-cache in [default

In [2]:
%load_ext sql

In [3]:
%sql spark

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/zorder_ex1 --recursive

In [4]:

df = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_201_99457.parquet")
df = df.select("customer_id", "category", "price", "quantity", "invoice_date")

In [5]:
display(df.limit(5))

+-----------+--------+------+--------+------------+
|customer_id|category|price |quantity|invoice_date|
+-----------+--------+------+--------+------------+
|201        |Clothing|900.24|3       |2021-07-04  |
|202        |Clothing|900.24|3       |2022-01-14  |
|203        |Toys    |35.84 |1       |2022-02-20  |
|204        |Clothing|1500.4|5       |2022-06-18  |
|205        |Souvenir|35.19 |3       |2022-04-27  |
+-----------+--------+------+--------+------------+



In [6]:
df_union = df
expected_rows = 20000000 # 20,000,000

while df_union.count() <= expected_rows:
    df_union = df_union.union(df_union)
    print(f"count: {df_union.count()}")

print(f"final count: {df_union.count()}")

count: 198514
count: 397028
count: 794056
count: 1588112
count: 3176224
count: 6352448
count: 12704896
count: 25409792
final count: 25409792


In [7]:

df_union.write.format("delta").mode("overwrite").saveAsTable("spark_catalog.deltalake.zorder_ex1")

In [8]:
%%time

spark.sql(
    """
    SELECT category,
        SUM(price * quantity) as total_sales
    FROM spark_catalog.deltalake.zorder_ex1
    WHERE customer_id = 201
    GROUP BY category
    """
)

CPU times: user 1.84 ms, sys: 274 μs, total: 2.12 ms
Wall time: 180 ms


DataFrame[category: string, total_sales: double]

In [9]:
%%sql

OPTIMIZE spark_catalog.deltalake.zorder_ex1
ZORDER BY customer_id;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/zorder_ex1,"Row(numFilesAdded=1, numFilesRemoved=256, filesAdded=Row(min=172182853, max=172182853, avg=172182853.0, totalFiles=1, totalSize=172182853), filesRemoved=Row(min=677463, max=677463, avg=677463.0, totalFiles=256, totalSize=173430528), partitionsOptimized=1, zOrderStats=Row(strategyName='all', inputCubeFiles=Row(num=0, size=0), inputOtherFiles=Row(num=256, size=173430528), inputNumCubes=0, mergedFiles=Row(num=256, size=173430528), numOutputCubes=1, mergedNumCubes=None), clusteringStats=None, numBins=1, numBatches=1, totalConsideredFiles=256, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1775038651924, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=5, numTableColumnsWithStats=5)"


In [0]:
# from delta.tables import DeltaTable

# table = DeltaTable.forName(spark, "spark_catalog.deltalake.zorder_ex1")
# table.optimize().executeZOrderBy("customer_id")

In [10]:
%%time

spark.sql(
    """
    SELECT category,
        SUM(price * quantity) as total_sales
    FROM spark_catalog.deltalake.zorder_ex1
    WHERE customer_id = 201
    GROUP BY category
    """
)

CPU times: user 547 μs, sys: 1.12 ms, total: 1.66 ms
Wall time: 44.6 ms


DataFrame[category: string, total_sales: double]

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/zorder_ex2 --recursive

In [11]:
#
df.write.format("delta").mode("overwrite").partitionBy("invoice_date").saveAsTable("spark_catalog.deltalake.zorder_ex2")

- Hive Style Partition: `invoice_date`
- ZORDER BY: `customer_id` 
- For each of those `invoice_date`

In [12]:
import pyspark.sql.functions as F

df_new_partition = df.filter(F.col("invoice_date") == "2023-01-01").withColumn("invoice_date", F.to_date(F.lit("2025-05-04")))


In [13]:

display(df_new_partition.limit(5))

+-----------+----------+-------+--------+------------+
|customer_id|category  |price  |quantity|invoice_date|
+-----------+----------+-------+--------+------------+
|986        |Clothing  |1500.4 |5       |2025-05-04  |
|1127       |Souvenir  |58.65  |5       |2025-05-04  |
|3271       |Shoes     |1800.51|3       |2025-05-04  |
|3485       |Technology|4200.0 |4       |2025-05-04  |
|4648       |Clothing  |1200.32|4       |2025-05-04  |
+-----------+----------+-------+--------+------------+



In [14]:

df_new_partition.write.format("delta").mode("append").partitionBy("invoice_date").saveAsTable("spark_catalog.deltalake.zorder_ex2")

In [15]:
%%sql

SELECT MAX(invoice_date) FROM spark_catalog.deltalake.zorder_ex2;

Running query in 'SparkSession'

1 rows affected.

Field 1
2025-05-04


In [16]:
%%sql

OPTIMIZE spark_catalog.deltalake.zorder_ex2 
WHERE invoice_date = '{current_day - 1}'
ZORDER BY customer_id;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
s3a://warehouse/default/deltalake/zorder_ex2,"Row(numFilesAdded=0, numFilesRemoved=0, filesAdded=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), filesRemoved=Row(min=None, max=None, avg=0.0, totalFiles=0, totalSize=0), partitionsOptimized=0, zOrderStats=Row(strategyName='all', inputCubeFiles=Row(num=0, size=0), inputOtherFiles=Row(num=0, size=0), inputNumCubes=0, mergedFiles=Row(num=0, size=0), numOutputCubes=0, mergedNumCubes=None), clusteringStats=None, numBins=0, numBatches=1, totalConsideredFiles=0, totalFilesSkipped=0, preserveInsertionOrder=False, numFilesSkippedToReduceWriteAmplification=0, numBytesSkippedToReduceWriteAmplification=0, startTimeMs=1775038786813, endTimeMs=0, totalClusterParallelism=8, totalScheduledTasks=0, autoCompactParallelismStats=None, deletionVectorStats=None, numTableColumns=5, numTableColumnsWithStats=5)"
